# Logistic Regression Playbook (80/20 Split)

This notebook mirrors your test flow in a fully playable format:

1. Load data with pandas
2. Split dataset into 80% train and 20% test
3. Preprocess in DataFrame form
4. Convert to NumPy arrays
5. Train and evaluate `LogisticRegressionModel`

Run cells from top to bottom.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make project root importable when notebook is opened from notebooks/
PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.domain.logistic_regression import LogisticRegressionModel

print("Project root:", PROJECT_ROOT)

Project root: /Users/akadilkalimoldayev/my_git/ecole42/42_dslr


In [3]:
DATASET_PATH = PROJECT_ROOT / "datasets" / "dataset_train.csv"

df = pd.read_csv(DATASET_PATH)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (1600, 19)
Columns: ['Index', 'Hogwarts House', 'First Name', 'Last Name', 'Birthday', 'Best Hand', 'Arithmancy', 'Astronomy', 'Herbology', 'Defense Against the Dark Arts', 'Divination', 'Muggle Studies', 'Ancient Runes', 'History of Magic', 'Transfiguration', 'Potions', 'Care of Magical Creatures', 'Charms', 'Flying']


,Index,Hogwarts House,First Name,Last Name,Birthday,Best Hand,Arithmancy,Astronomy,Herbology,Defense Against the Dark Arts,Divination,Muggle Studies,Ancient Runes,History of Magic,Transfiguration,Potions,Care of Magical Creatures,Charms,Flying
0,0,Ravenclaw,Tamara,Hsu,2000-03-30,Left,58384.0,-487.886086,5.727180,4.878861,4.722,272.035831,532.484226,5.231058,1039.788281,3.790369,0.715939,-232.79405,-26.89
1,1,Slytherin,Erich,Paredes,1999-10-14,Right,67239.0,-552.060507,-5.987446,5.520605,-5.612,-487.340557,367.760303,4.107170,1058.944592,7.248742,0.091674,-252.18425,-113.45
2,2,Ravenclaw,Stephany,Braun,1999-11-03,Left,23702.0,-366.076117,7.725017,3.660761,6.140,664.893521,602.585284,3.555579,1088.088348,8.728531,-0.515327,-227.34265,30.42
3,3,Gryffindor,Vesta,Mcmichael,2000-08-19,Left,32667.0,697.742809,-6.497214,-6.977428,4.026,-537.001128,523.982133,-4.809637,920.391449,0.821911,-0.014040,-256.84675,200.64
4,4,Gryffindor,Gaston,Gibbs,1998-09-27,Left,60158.0,436.775204,-7.820623,NaN,2.236,-444.262537,599.324514,-3.444377,937.434724,4.311066,-0.264070,-256.38730,157.98


## 1) Select target and numeric feature columns

These are the same 13 subject columns used in your tests.

In [4]:
TARGET_COLUMN = "Hogwarts House"
FEATURE_COLUMNS = [
    "Herbology",
    "Defense Against the Dark Arts",
    "Divination",
    "Muggle Studies",
    "Ancient Runes",
    "History of Magic",
    "Transfiguration",
    "Potions",
    "Care of Magical Creatures",
    "Charms",
    "Flying",
]

X_df = df[FEATURE_COLUMNS].copy()
y_series = df[TARGET_COLUMN].copy()

print("X_df shape:", X_df.shape)
print("y shape:", y_series.shape)
print("Classes:", sorted(y_series.unique().tolist()))

X_df shape: (1600, 11)
y shape: (1600,)
Classes: ['Gryffindor', 'Hufflepuff', 'Ravenclaw', 'Slytherin']


In [5]:
# Deterministic 80/20 split
SEED = 42
TRAIN_RATIO = 0.95

shuffled_idx = X_df.sample(frac=1.0, random_state=SEED).index
split_idx = int(len(shuffled_idx) * TRAIN_RATIO)

train_idx = shuffled_idx[:split_idx]
test_idx = shuffled_idx[split_idx:]

X_train_df = X_df.loc[train_idx].copy()
X_test_df = X_df.loc[test_idx].copy()
y_train_series = y_series.loc[train_idx].copy()
y_test_series = y_series.loc[test_idx].copy()

print("Train size:", len(X_train_df))
print("Test size:", len(X_test_df))

Train size: 1520
Test size: 80


## 2) Impute missing values from train means and convert to ndarray

This keeps preprocessing leakage-safe: test rows are filled using train statistics only.

In [6]:
train_means = X_train_df.mean(numeric_only=True)

X_train_df = X_train_df.fillna(train_means)
X_test_df = X_test_df.fillna(train_means)

X_train = X_train_df.to_numpy(dtype=float)
X_test = X_test_df.to_numpy(dtype=float)
y_train = y_train_series.to_numpy(dtype=str)
y_test = y_test_series.to_numpy(dtype=str)

print("X_train ndarray:", X_train.shape, X_train.dtype)
print("X_test ndarray:", X_test.shape, X_test.dtype)
print("y_train ndarray:", y_train.shape, y_train.dtype)
print("y_test ndarray:", y_test.shape, y_test.dtype)

X_train ndarray: (1520, 11) float64
X_test ndarray: (80, 11) float64
y_train ndarray: (1520,) <U10
y_test ndarray: (80,) <U10


In [7]:
model = LogisticRegressionModel()
model.fit(X_train, y_train)

print("Model trained.")
print("weights shape:", model.weights.shape)
print("bias shape:", model.bias.shape)
print("labels:", model.labels)

Model trained.
weights shape: (11, 4)
bias shape: (4,)
labels: ['Gryffindor' 'Hufflepuff' 'Ravenclaw' 'Slytherin']


In [8]:
predictions = model.predict(X_test)
accuracy = LogisticRegressionModel.compare_predictions(predictions, y_test)

print(f"Accuracy: {accuracy:.4f}")

results_df = pd.DataFrame(
    {
        "truth": y_test,
        "prediction": predictions,
    }
)
results_df.head(10)

Accuracy: 0.9875


,truth,prediction
0,Hufflepuff,Hufflepuff
1,Ravenclaw,Ravenclaw
2,Slytherin,Slytherin
3,Hufflepuff,Hufflepuff
4,Hufflepuff,Hufflepuff
5,Ravenclaw,Ravenclaw
6,Hufflepuff,Hufflepuff
7,Ravenclaw,Ravenclaw
8,Gryffindor,Gryffindor
9,Hufflepuff,Hufflepuff


In [ ]:
model.save_json("./logistic_regression_model.json")

In [ ]:
new_model = LogisticRegressionModel.from_json("./logistic_regression_model.json")
new_predictions = new_model.predict(X_test)
new_accuracy = LogisticRegressionModel.compare_predictions(new_predictions, y_test)
print(f"New accuracy: {new_accuracy:.4f}")

New accuracy: 0.9875
